In [ ]:
# Install dependencies
!pip install huggingface_hub datasets pandas scikit-learn wandb tqdm torch transformers bitsandbytes accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 11.4 MB/s eta 0:00:00


In [ ]:
import wandb
from google.colab import userdata

# Login to W&B
!wandb login --relogin

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [ ]:
import os
import tarfile
import urllib.request
import pandas as pd
import random
import numpy as np
from glob import glob
from tqdm import tqdm

# HOUSE RULE: REPRODUCIBILITY
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

#  Download & Extract Stanford Data (Same as Day 1)
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
tar_name = "aclImdb_v1.tar.gz"

if not os.path.exists("aclImdb"):
    print(f" Downloading {tar_name} (80MB)...")
    urllib.request.urlretrieve(url, tar_name)
    print(" Extracting (this takes a moment)...")
    with tarfile.open(tar_name, "r:gz") as tar:
        tar.extractall()


 Extracting (this takes a moment)...


/tmp/ipython-input-3714136287.py:24: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


In [ ]:
# Loader Function (Directly from Day 1 Logic)
def load_raw_imdb(data_dir='aclImdb'):
    data = {'train': [], 'test': []}
    for split in ['train', 'test']:
        for sentiment, label in [('pos', 1), ('neg', 0)]:
            path = os.path.join(data_dir, split, sentiment, '*.txt')
            files = glob(path)
            # We don't need ALL files for Day 2, but let's load a subset to be safe
            # or load all if you want strict parity. Loading all takes ~1 min in Colab.
            for file_path in tqdm(files, desc=f"Loading {split}/{sentiment}", leave=False):
                with open(file_path, 'r', encoding='utf-8') as f:
                    data[split].append({'text': f.read(), 'label': label})

    # Shuffle with SEED
    train_df = pd.DataFrame(data['train']).sample(frac=1, random_state=SEED).reset_index(drop=True)
    test_df = pd.DataFrame(data['test']).sample(frac=1, random_state=SEED).reset_index(drop=True)
    return train_df, test_df

print(" Loading Data from Raw Files")
train_full, test_full = load_raw_imdb()

 Loading Data from Raw Files


In [ ]:
# Create Small Sets for Prompting (Evaluation)
# Eval Set: 50 random samples from TEST
eval_df = test_full.sample(n=50, random_state=SEED).reset_index(drop=True)

# Shot Bank: 10 random samples from TRAIN (for few-shot examples)
shot_bank = train_full.sample(n=10, random_state=SEED).reset_index(drop=True)

print(f" Data Ready (Stanford Source): {len(eval_df)} eval samples, {len(shot_bank)} few-shot examples.")
print(f"Sample Text: {eval_df.iloc[0]['text'][:100]}...")

 Data Ready (Stanford Source): 50 eval samples, 10 few-shot examples.
Sample Text: I recently rented Twister, a movie I'd seen several years ago on TV, and it has aged well; I found m...


In [ ]:
test_full.sample(5)

,text,label
6868,"I recently rented Twister, a movie I'd seen se...",1
24016,"I am a big fan of ""Auntie Mame"" with Rosalind ...",0
9668,i consider this movie as one of the most inter...,1
13640,I'm at a loss. This entire movie made absolute...,0
14018,I very nearly did not see 'Hi-De-Hi!'. I think...,1


### **LLM Client Setup**

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

print("Downloading Qwen-0.5B fast using HF mirror")

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [ ]:
# 2. Create Pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50,
    return_full_text=False
)

print("Micro-Model Loaded Locally")

Device set to use cuda:0


Micro-Model Loaded Locally


In [ ]:
# 3. Define the Local LLM Function
def call_llm(prompt, temperature=0.1):
    messages = [{"role": "user", "content": prompt}]

    prompt_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    try:
        outputs = pipe(
            prompt_formatted,
            max_new_tokens=50,
            do_sample=True,
            temperature=temperature,
            top_k=50,
            top_p=0.95
        )
        return outputs[0]["generated_text"].strip()
    except Exception as e:
        return "ERROR"

In [ ]:
# Test
print(f"Test Response: {call_llm('Sentiment of this: I loved it. Label:')}")

Test Response: The sentiment expressed in the phrase "I loved it" is positive and enthusiastic. The use of the word "loved" indicates that the speaker had a favorable or very positive experience with something or someone. This kind of language often carries a strong positive


### **The Prompt Playbook**

In [ ]:
def generate_prompt(text, strategy, shot_bank):
  """
  Constructs the prompt based on chosen strategy
  """
  text= text[:8000] # truncating to avoid context limit
  # 1. Zero-Shot Basic
  if strategy== "Zero-Shot Basic":
    return f"""Classify the sentiment of this movie review as "Positive" or "Negative"
    Review: "{text}"
    Sentiment: """

  # 2. Zero-Shot Persona
  elif strategy== "Zero-Shot Persona":
    return f"""You are an expert film critic. Analyze the following review.
    If the reviewer liked the movie, say "Positive". If they disliked it, say "Negative".
    Review: "{text}"
    Sentiment:"""

  # 3. Few-Shot (1 Example)
  elif strategy== "Few-Shot (1-shot)":
    ex= shot_bank.iloc[0]
    ex_label = 'Positive' if ex['label'] == 1 else 'Negative'
    return f"""Task: Classify Movie Reviews.
    Example:
    Review: "{ex['text'][:200]}..."
    Sentiment: {ex_label}

    Review: "{text}"
    Sentiment:"""

  # # 4. Few-Shot (3 Examples)
  elif strategy == "Few-Shot (3-shot)":
      examples = ""
      for i in range(3):
            ex = shot_bank.iloc[i]
            lbl = "Positive" if ex['label'] == 1 else "Negative"
            examples += f'Review: "{ex["text"][:150]}..."\nSentiment: {lbl}\n\n'
      return f"""Task: Classify Movie Reviews.
{examples}
Review: "{text}"
Sentiment:"""


    # 5. Chain of Thought (CoT)
  elif strategy == "Chain of Thought":
        return f"""Analyze the movie review.
Step 1: Identify emotional keywords.
Step 2: Decide if the tone is Positive or Negative.
Format:
Reasoning: [Reasoning]
Sentiment: [Label]

Review: "{text}"
"""
  return "Error"


## **The Ablation Experiment Loop**

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
import time

# 1. Initialize W&B Run with config
temperatures = [0.1, 0.7] # 0.1 = Strict, 0.7 = Creative
strategies = [
    "Zero-Shot Basic",
    "Zero-Shot Persona",
    "Few-Shot (1-shot)",
    "Few-Shot (3-shot)",
    "Chain of Thought"
]


In [ ]:
# 2. Initialize W&B Run
import wandb
run = wandb.init(
    project="Week-4",
    job_type="full-ablation-study",
    name="day2-all-strategies-with-temps",
    config={
        "n_samples": len(eval_df),
        "strategies": strategies,
        "temperatures": temperatures
    }
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Currently logged in as: nishanttg (nishantg) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# 3. Create W&B Table
table_columns = ["id", "strategy", "temperature", "review_snippet", "true_label", "predicted_label", "is_correct", "latency", "raw_response"]
results_table = wandb.Table(columns=table_columns)

In [ ]:
# 4. Storage for Metrics
# Key format: "Strategy_Temp" (e.g., "Zero-Shot Basic_0.1")
experiment_data = {}
print(f"Starting Massive Evaluation: {len(strategies)} Strategies x {len(temperatures)} Temps x {len(eval_df)} Samples")

Starting Massive Evaluation: 5 Strategies x 2 Temps x 50 Samples


In [ ]:
# OUTER LOOP: TEMPERATURE
for temp in temperatures:
    print(f"\n Setting Temperature to: {temp}")

    #  MIDDLE LOOP: STRATEGY
    for strategy in strategies:
        key = f"{strategy}_T{temp}"
        experiment_data[key] = {'y_true': [], 'y_pred': [], 'latencies': []}

        print(f" Testing Strategy: {strategy}...")

        #  INNER LOOP: DATA SAMPLES
        for i, row in tqdm(eval_df.iterrows(), total=len(eval_df), leave=False):
            text = row['text']
            true_label = "Positive" if row['label'] == 1 else "Negative"

            # A. Generate Prompt
            prompt = generate_prompt(text, strategy, shot_bank)

            # B. Call Model (Passing the specific Temperature)
            start_t = time.time()
            raw_response = call_llm(prompt, temperature=temp)
            latency = time.time() - start_t

            # C. Parse Prediction
            # Logic: Look for keywords. If both or neither appear, mark Unknown.
            pred = "Unknown"
            lower_response = raw_response.lower()
            if "positive" in lower_response and "negative" not in lower_response:
                pred = "Positive"
            elif "negative" in lower_response and "positive" not in lower_response:
                pred = "Negative"
            # Fallback for CoT or mixed responses: prioritize the first occurrence or specific formatting if needed
            elif "positive" in lower_response:
                pred = "Positive" # Naive fallback
            elif "negative" in lower_response:
                pred = "Negative" # Naive fallback

            # D. Store Data
            experiment_data[key]['y_true'].append(true_label)
            experiment_data[key]['y_pred'].append(pred)
            experiment_data[key]['latencies'].append(latency)

            # E. Add to Table
            results_table.add_data(
                i, strategy, temp, text[:100] + "...",
                true_label, pred, (pred == true_label),
                latency, raw_response
            )

            # Rate Limit Safety
            time.sleep(0.5)


 Setting Temperature to: 0.1
 Testing Strategy: Zero-Shot Basic...


 18%|█▊        | 9/50 [00:25<01:36,  2.36s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


 Testing Strategy: Zero-Shot Persona...


 Testing Strategy: Few-Shot (1-shot)...


 Testing Strategy: Few-Shot (3-shot)...


...


 Setting Temperature to: 0.7
 Testing Strategy: Zero-Shot Basic...


 Testing Strategy: Zero-Shot Persona...


 Testing Strategy: Few-Shot (1-shot)...


 Testing Strategy: Few-Shot (3-shot)...


 Testing Strategy: Chain of Thought...


In [ ]:
# 5. Calculate Final Metrics & Log
print("\n Final Analysis:")
wandb.log({"full_results_table": results_table})

for key, data in experiment_data.items():
    # Calculate Metrics
    # We use labels=["Positive", "Negative"] to strictly penalize "Unknown" predictions
    f1 = f1_score(data['y_true'], data['y_pred'], labels=["Positive", "Negative"], average='macro')
    acc = accuracy_score(data['y_true'], data['y_pred'])
    avg_lat = sum(data['latencies']) / len(data['latencies'])

    # Print to Console
    print(f"Run: {key:35} | F1: {f1:.4f} | Acc: {acc:.4f} | Lat: {avg_lat:.2f}s")


 Final Analysis:
Run: Zero-Shot Basic_T0.1                | F1: 0.8667 | Acc: 0.8600 | Lat: 2.08s
Run: Zero-Shot Persona_T0.1              | F1: 0.6435 | Acc: 0.6800 | Lat: 0.15s
Run: Few-Shot (1-shot)_T0.1              | F1: 0.8572 | Acc: 0.8600 | Lat: 0.11s
Run: Few-Shot (3-shot)_T0.1              | F1: 0.4544 | Acc: 0.5600 | Lat: 1.20s
Run: Chain of Thought_T0.1               | F1: 0.6729 | Acc: 0.5800 | Lat: 1.83s
Run: Zero-Shot Basic_T0.7                | F1: 0.8367 | Acc: 0.8200 | Lat: 1.70s
Run: Zero-Shot Persona_T0.7              | F1: 0.7211 | Acc: 0.7400 | Lat: 0.43s
Run: Few-Shot (1-shot)_T0.7              | F1: 0.8164 | Acc: 0.8200 | Lat: 0.15s
Run: Few-Shot (3-shot)_T0.7              | F1: 0.6435 | Acc: 0.6800 | Lat: 0.96s
Run: Chain of Thought_T0.7               | F1: 0.7328 | Acc: 0.6400 | Lat: 1.88s


In [ ]:
# 1. Start a new "Reporting" run (since the previous one finished)
run = wandb.init(
    project="Week-4",
    name="day2-metrics",
    job_type="upload-metrics"
)

print(" Re-logging Metrics ")

# 2. Create a clean Summary Table (Best for your report!)
summary_table = wandb.Table(columns=["Strategy", "Temperature", "F1_Macro", "Accuracy", "Latency"])

# 3. Iterate through the EXISTING data in memory
for key, data in experiment_data.items():

    # Recalculate metrics (Fast)
    f1 = f1_score(data['y_true'], data['y_pred'], labels=["Positive", "Negative"], average='macro')
    acc = accuracy_score(data['y_true'], data['y_pred'])
    avg_lat = sum(data['latencies']) / len(data['latencies'])

    # Parse the key (e.g. "Zero-Shot Basic_T0.1") to look nice in the table
    try:
        strategy_name, temp_val = key.split("_T")
    except:
        strategy_name = key
        temp_val = "N/A"

    # --- FIX: Log INSIDE the loop ---
    wandb.log({
        f"f1_{key}": f1,
        f"acc_{key}": acc,
        f"lat_{key}": avg_lat
    })

    # Add to the summary table
    summary_table.add_data(strategy_name, temp_val, f1, acc, avg_lat)

    print(f" Logged {key}")

# 4. Upload the Summary Table
wandb.log({"final_leaderboard": summary_table})

print("\n Re-Logged")
wandb.finish()

 Re-logging Metrics 
 Logged Zero-Shot Basic_T0.1
 Logged Zero-Shot Persona_T0.1
 Logged Few-Shot (1-shot)_T0.1
 Logged Few-Shot (3-shot)_T0.1
 Logged Chain of Thought_T0.1
 Logged Zero-Shot Basic_T0.7
 Logged Zero-Shot Persona_T0.7
 Logged Few-Shot (1-shot)_T0.7
 Logged Few-Shot (3-shot)_T0.7
 Logged Chain of Thought_T0.7

 Re-Logged


acc_Chain of Thought_T0.1,▁
acc_Chain of Thought_T0.7,▁
acc_Few-Shot (1-shot)_T0.1,▁
acc_Few-Shot (1-shot)_T0.7,▁
acc_Few-Shot (3-shot)_T0.1,▁
acc_Few-Shot (3-shot)_T0.7,▁
acc_Zero-Shot Basic_T0.1,▁
acc_Zero-Shot Basic_T0.7,▁
acc_Zero-Shot Persona_T0.1,▁
acc_Zero-Shot Persona_T0.7,▁
+20,...


### **zephyr**

In [ ]:

import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- HOUSE RULE: SEEDS ---
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(" Downloading Zephyr-7b (Local GPU mode)")

# 1. Configure 4-bit quantization (Fits on free Colab GPU)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
# 2. Load Tokenizer & Model
model_id = "HuggingFaceH4/zephyr-7b-beta"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model-00008-of-00008.safetensors:   0%|          | 0.00/816M [00:00<?, ?B/s]

model-00003-of-00008.safetensors:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

model-00004-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [ ]:
# 3. Create Pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50, # Limit output length
    return_full_text=False
)

print("Model Loaded Locally")

Device set to use cuda:0


Model Loaded Locally


In [ ]:
# 4. Define the Local LLM Function
def call_llm(prompt, temperature=0.1):
    # Format input as chat
    messages = [{"role": "user", "content": prompt}]
    prompt_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Generate
    outputs = pipe(
        prompt_formatted,
        max_new_tokens=50,
        do_sample=True,
        temperature=temperature,
        top_k=50,
        top_p=0.95
    )

    # Extract text
    return outputs[0]["generated_text"].strip()

In [ ]:
# Test
print(f"Test Response: {call_llm('Hello')}")

Test Response: Hello! I'm glad you're here. If you have any questions, requests, or just want to chat, feel free to let me know. I'm here to help you out in any way I can. Whether you need assistance


In [ ]:
from sklearn.metrics import f1_score, accuracy_score
import time

# 1. Configuration
strategies = [
    "Zero-Shot Basic",
    "Zero-Shot Persona",
    "Few-Shot (1-shot)",
    "Few-Shot (3-shot)",
    "Chain of Thought"
]
temperatures = [0.1, 0.7] # 0.1 = Strict, 0.7 = Creative


In [ ]:
# 2. Initialize W&B Run
run = wandb.init(
    project="Week-4",
    job_type="full-ablation-study",
    name="day2-zephyr",
    config={
        "n_samples": len(eval_df),
        "strategies": strategies,
        "temperatures": temperatures
    }
)

In [ ]:
# 3. Create W&B Table
table_columns = ["id", "strategy", "temperature", "review_snippet", "true_label", "predicted_label", "is_correct", "latency", "raw_response"]
results_table = wandb.Table(columns=table_columns)

In [ ]:
# 4. Storage for Metrics
# Key format: "Strategy_Temp" (e.g., "Zero-Shot Basic_0.1")
experiment_data = {}

print(f" Starting Massive Evaluation: {len(strategies)} Strategies x {len(temperatures)} Temps x {len(eval_df)} Samples")

#  OUTER LOOP: TEMPERATURE
for temp in temperatures:
    print(f"\n Setting Temperature to: {temp}")

    #  MIDDLE LOOP: STRATEGY
    for strategy in strategies:
        key = f"{strategy}_T{temp}"
        experiment_data[key] = {'y_true': [], 'y_pred': [], 'latencies': []}

        print(f"   Testing Strategy: {strategy}...")

        # INNER LOOP: DATA SAMPLES
        for i, row in tqdm(eval_df.iterrows(), total=len(eval_df), leave=False):
            text = row['text']
            true_label = "Positive" if row['label'] == 1 else "Negative"

            # A. Generate Prompt
            prompt = generate_prompt(text, strategy, shot_bank)

            # B. Call Model (Passing the specific Temperature)
            start_t = time.time()
            raw_response = call_llm(prompt, temperature=temp)
            latency = time.time() - start_t

            # C. Parse Prediction
            # Logic: Looking for keywords. If both or neither appear, mark Unknown.
            pred = "Unknown"
            lower_response = raw_response.lower()
            if "positive" in lower_response and "negative" not in lower_response:
                pred = "Positive"
            elif "negative" in lower_response and "positive" not in lower_response:
                pred = "Negative"
            # Fallback for CoT or mixed responses: prioritize the first occurrence or specific formatting if needed
            elif "positive" in lower_response:
                pred = "Positive" # Naive fallback
            elif "negative" in lower_response:
                pred = "Negative" # Naive fallback

            # D. Store Data
            experiment_data[key]['y_true'].append(true_label)
            experiment_data[key]['y_pred'].append(pred)
            experiment_data[key]['latencies'].append(latency)

            # E. Add to Table
            results_table.add_data(
                i, strategy, temp, text[:100] + "...",
                true_label, pred, (pred == true_label),
                latency, raw_response
            )

            # Rate Limit Safety
            time.sleep(0.5)

 Starting Massive Evaluation: 5 Strategies x 2 Temps x 50 Samples

 Setting Temperature to: 0.1
   Testing Strategy: Zero-Shot Basic...


   Testing Strategy: Zero-Shot Persona...


   Testing Strategy: Few-Shot (1-shot)...


   Testing Strategy: Few-Shot (3-shot)...


   Testing Strategy: Chain of Thought...



 Setting Temperature to: 0.7
   Testing Strategy: Zero-Shot Basic...


   Testing Strategy: Zero-Shot Persona...


   Testing Strategy: Few-Shot (1-shot)...


   Testing Strategy: Few-Shot (3-shot)...


   Testing Strategy: Chain of Thought...


In [ ]:
# 5. Calculate Final Metrics & Log
print("\n Final Analysis:")
wandb.log({"full_results_table": results_table})

for key, data in experiment_data.items():
    # Calculate Metrics
    # We use labels=["Positive", "Negative"] to strictly penalize "Unknown" predictions
    f1 = f1_score(data['y_true'], data['y_pred'], labels=["Positive", "Negative"], average='macro')
    acc = accuracy_score(data['y_true'], data['y_pred'])
    avg_lat = sum(data['latencies']) / len(data['latencies'])

    # Print to Console
    print(f"Run: {key:35} | F1: {f1:.4f} | Acc: {acc:.4f} | Lat: {avg_lat:.2f}s")

    # Log to W&B Charts
    wandb.log({
        f"f1_{key}": f1,
        f"acc_{key}": acc,
        f"lat_{key}": avg_lat
    })
wandb.finish()


 Final Analysis:
Run: Zero-Shot Basic_T0.1                | F1: 0.8650 | Acc: 0.8600 | Lat: 3.47s
Run: Zero-Shot Persona_T0.1              | F1: 0.8188 | Acc: 0.8000 | Lat: 3.34s
Run: Few-Shot (1-shot)_T0.1              | F1: 0.7749 | Acc: 0.7200 | Lat: 3.85s
Run: Few-Shot (3-shot)_T0.1              | F1: 0.7888 | Acc: 0.7800 | Lat: 4.62s
Run: Chain of Thought_T0.1               | F1: 0.8006 | Acc: 0.7000 | Lat: 3.66s
Run: Zero-Shot Basic_T0.7                | F1: 0.9282 | Acc: 0.9200 | Lat: 3.04s
Run: Zero-Shot Persona_T0.7              | F1: 0.8387 | Acc: 0.8000 | Lat: 3.45s
Run: Few-Shot (1-shot)_T0.7              | F1: 0.7381 | Acc: 0.6400 | Lat: 3.64s
Run: Few-Shot (3-shot)_T0.7              | F1: 0.7408 | Acc: 0.7000 | Lat: 3.77s
Run: Chain of Thought_T0.7               | F1: 0.6218 | Acc: 0.4800 | Lat: 3.69s


acc_Chain of Thought_T0.1,▁
acc_Chain of Thought_T0.7,▁
acc_Few-Shot (1-shot)_T0.1,▁
acc_Few-Shot (1-shot)_T0.7,▁
acc_Few-Shot (3-shot)_T0.1,▁
acc_Few-Shot (3-shot)_T0.7,▁
acc_Zero-Shot Basic_T0.1,▁
acc_Zero-Shot Basic_T0.7,▁
acc_Zero-Shot Persona_T0.1,▁
acc_Zero-Shot Persona_T0.7,▁
+20,...
